## Build an agent

In [ ]:
import os
# os.getcwd()
os.chdir("/Users/kay/Desktop/job-match-hackathon")
print("New directory:", os.getcwd())

In [ ]:
# Import relevant functionality
# from langchain_anthropic import ChatAnthropic
# from langchain_community.tools.tavily_search import TavilySearchResults
from dotenv import load_dotenv
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import MemorySaver
from langgraph.prebuilt import create_react_agent
from langchain_community.chat_models import ChatOllama
from langchain_ollama.llms import OllamaLLM
import getpass
from dotenv import load_dotenv

import importlib
import src.query_engine  # your module


In [ ]:
# Read the contents of a text file
with open("output.txt", "r", encoding="utf-8") as f:
    output = f.read()

print(output)

In [ ]:
import json
output_dict = json.loads(output)

In [ ]:
import csv
with open("job_matches.csv", "w", newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=output_dict[0].keys())
    writer.writeheader()
    writer.writerows(output_dict)

In [ ]:
# %pip install -qU langchain-text-splitters

In [ ]:
from langchain_text_splitters import MarkdownHeaderTextSplitter

headers_to_split_on = [
    ("###", "Job Recommendation")
]

markdown_splitter = MarkdownHeaderTextSplitter(headers_to_split_on)
md_header_splits = markdown_splitter.split_text(output)
md_header_splits


In [ ]:
import csv

# Assuming `doc_list` is your list of Document objects
with open("documents_output.csv", mode="w", newline="", encoding="utf-8") as file:
    writer = csv.writer(file)
    writer.writerow(["Job Title", "Evaluation"])  # Header

    for doc in md_header_splits:
        writer.writerow([value for key, value in doc.metadata.items()] + [doc.page_content])

In [ ]:
# %%capture --no-stderr
# %pip install --quiet -U langchain langchain_community tiktoken langchain-mistralai scikit-learn langgraph tavily-python bs4



### Initiate Langsmith

In [ ]:
import os, getpass

def _set_env(var: str):
    if not os.environ.get(var):
        os.environ[var] = getpass.getpass(f"{var}: ")

_set_env("MISTRAL_API_KEY")
# _set_env("TAVILY_API_KEY")
os.environ['TOKENIZERS_PARALLELISM'] = 'true'

In [ ]:
# Load environment variables from .env file
load_dotenv()

### Define tools of index and query

In [ ]:
from src.query_engine import QueryEngine
from src.tools import tools

In [ ]:
# tools = [tools]

### Instantiate agent

In [ ]:
from langchain.chat_models import init_chat_model

model = init_chat_model("mistral-small-latest", model_provider="mistralai")

In [ ]:
from langchain_core.messages import HumanMessage

# response = model.invoke([HumanMessage(content="hi!")])
# response.content

### Function calling

In [ ]:
importlib.reload(src.query_engine)
from src.query_engine import QueryEngine

tools = [QueryEngine]

tools

In [ ]:
model_with_tools = model.bind_tools(tools)

In [ ]:
# response = model_with_tools.invoke([HumanMessage(content="YIJIN BAO\n\n\t\t+1-773-490-7200\n\n\t\tChicago, Illinois\n\n\t\tkay062@uchicago.edu, LinkedIn | GitHub\n\n\n\n\n\neducation\n\n\tHarris School of Public Policy, University of Chicago\tChicago, IL\n\n\tData Analytics Specialization of Master of Public Policy (MPP), Graduate with Honor\tJune 2024\n\nData Science: ETL pipeline, NLP, Deep Learning, Machine Learning, Computer Vision\n\nData Analytics: data collection, management, cleaning, manipulation, sanity check, analysis, visualization, GIS\n\nPublic Policy analysis: modeling and research for transportation policy, healthcare policy, urban policy\n\nBusiness data product: dashboard design, interactive chatbot, database management, automated workflows\n\n\n\n\tShanghai International Studies University\tShanghai, China\n\n\tDual Bachelor of Laws and English, Outstanding Undergraduate Thesis Honor\tJuly 2022\n\nInternational Law; International Economics Law; International Finance Law\n\nHonored Student Scholarship in 2018-2019 Academic Year\n\nStudent Union President (2020-2021): maintained alumni network and organize career networking events\n\n\n\nexperience\n\n\tUnited nations statisticS DIVISION\tHybrid\n\n\tStatistical Intern using Python\t February 2025 ? June 2025\n\nDeveloped a web-scraping program using Amazon Bedrock, called Claude Sonnet 3 model and engineered prompt to optimize AI?s performance of the accurate collection of unstructured event data from international organizations? websites\n\nAutomated and the scraping and processing of user interaction data of Global Network of Data Officers and Statisticians, continuously refining the process and analyzing participation trends to enhance outreach and network management\n\n\n\nIllinois Criminal Justice information authority\t                                                                               Chicago, IL\n\n\tState Government Agency | Research Analysis Intern using R, SQL\tOct 2024 ? Mar 2025, June 2023 ? Aug 2023\n\nEngineered a highly automated data pipeline to efficiently extract from PDF, convert to R-readable format and integrate 50 years of federal disposition data, pre-processed data for analysis and conduct sanity check with complete documentation  \n\nCleaned, validated, and transformed a 20GB+ Illinois Traffic Stop dataset (2004?2022) to ensure data quality, standardize formats, and generate performance metrics, enabling statistical analysis of equity in police pull-overs, including visualizing hit rates and false alarm rates by race and police agency as part of the Veil of Darkness analysis\n\nAudited data of Illinois Death in Custody Reporting from 2019 to 2024 by developing a program to automate matching and identify unreported incidents through cross-referencing manually collected media-reported cases \n\n\n\n\tRural Advancement Foundation International (RAFI)\tChicago, IL\n\n\tNGO for Economic Justice | Data science fellow at Data Science Clinic using Python\t Jan 2024 ? Mar 2024\n\nDeveloped and maintained a Streamlit dashboard to visualize 30 years of market trends across 3,455 auction houses, supporting RAFI?s efforts to convey findings to a Federal Government Agency for digitalization of contracts\n\nImproved a computer vision model?s poultry estimation accuracy by ~10% (~2,000 barns) using buffer analysis with Google Earth Engine, enhancing RAFI?s dashboard to capture market trends of 20,000+ poultry barns in North Carolina\n\nPresented a technical brief outlining data merging strategies, dashboard development, market capture and model enhancements to provide the client with a clear understanding of technical product and analysis results\n\n\n\n\tIllinois department of employment safety\tChicago, IL\n\n\tState Employment Government Agency | Data analytics fellow at Policy Lab using R, Google Suites\t Sept 2023 ? Dec 2023\n\nSupported clients? promotion of Unemployment Insurance equity by using SQL to extract datasets on 12,000 employees and 2,000+ applicants to create a merged individual level dataset with employment flux and application information\n\nDeveloped a metrics to identify the gap in laid-off workers? participation in Unemployment Insurance and visualize it in Google Sheet and Slides to evaluate sectorial participation?region, industry, union vs. non-union, and company size\n\nCoordinated timely communications with the client on refining government outreach strategies and preventing further applicant attrition by maintaining information updates and delivering high-quality policy memo and presentation\n\n\n\n\n\nResearch publication\n\n\tKuda-Singappulige, G., Green, E., Reichert, J., Bao Y., & Gruschow, K. (2025). Veil of darkness analysis of Illinois traffic stop data. Illinois Criminal Justice Information Authority.\n\nskills\n\n\n\nMS Office Suite: Word, Excel, Power Point, One Driver, Teams\n\nAI solutions: API integration, AWS cloud, prompt engineering, UI/UX design, data migration, deployment\n\nMachine Learning: feature engineering, dimensionality reduction, regularization, tree-based models, SVM, clustering, KNN\n\nData Visualization: Tableau, R(Shiny, sf, tmap, ggspatial, ggplot2), Python(Streamlit, GeoPandas, Seaborn, Matplotlib)\n\nNLP: Tokenization, summarization, embeddings, sentiment analysis, knowledge graphs, named entity recognition\n\nSQL, PostgreSQL: data management, filtering, aggregation, joining, extracting, interacting with R and Python\n\nPython (Boto, Langchain, Openai LLamaIndex, Pandas, NumPy, SpaCy, BeautifulSoup, Selenium)\n\nR (data.table, date.time, dplyr, tidyverse, plotly, viridis, Shiny)\n\nMS Office Suite: Word, Excel, Power Point, One Driver, Teams")])

# print(f"ContentString: {response.content}")
# print(f"ToolCalls: {response.tool_calls}")

### Create the agent

In [ ]:
from langgraph.prebuilt import create_react_agent

agent_executor = create_react_agent(model, tools)

### Run the agent

#### Here should be replaced with front end upload: research how

In [ ]:
from langchain_community.document_loaders import Docx2txtLoader
loader = Docx2txtLoader("resume.docx")

data = loader.load()

# Convert to a single string
doc_text = "\n".join([doc.page_content for doc in data])

doc_text 

In [ ]:
prompt = f"""This is my resume. Please help me rank them based on my education, skills, and experience.

Resume:
{doc_text}
"""

In [ ]:
for step in agent_executor.stream(
    {"messages": [HumanMessage(content=prompt)]},
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

### Add in memory

In [ ]:
from langgraph.checkpoint.memory import MemorySaver

memory = MemorySaver()

In [ ]:
agent_executor = create_react_agent(model, tools, checkpointer=memory)

config = {"configurable": {"thread_id": "abc123"}}

In [ ]:
for chunk in agent_executor.stream(
    {"messages": [HumanMessage(content=doc_text)]}, config
):
    print(chunk)
    print("----")